# Fixed-100 v2 Diagnostics: Metrics, Autocorrelation, ESS/R-hat, and LDA

This notebook analyzes the short-chain fixed-100 experiments for:

- datasets: Abalone, Concrete, Friedman
- swap settings: `swap_sigma=True` and `swap_sigma=False`
- methods: `default`, `default_pt`, `mtmh`, `mtmh_pt`

The main goal is to systematically test whether swapping `sigma2` produces abnormal peaks / poor mixing, and whether `swap_sigma=False` behaves better.

Formal metrics used here match the project notes:

\[
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^n (y_i-\hat y_i)^2}
\]

\[
\mathrm{CRPS}(F,y)=\mathbb{E}|X-y|-\frac{1}{2}\mathbb{E}|X-X'|
\]

\[
\rho_k=\frac{\mathrm{Cov}(X_t,X_{t+k})}{\mathrm{Var}(X_t)}
\]

\[
\mathrm{ESS}=\frac{N}{1+2\sum_{k=1}^{\infty}\rho_k}
\]

For \(\hat R\), this notebook uses ArviZ's rank-normalized R-hat when ArviZ is installed. If ArviZ is not installed, it falls back to the classical Gelman--Rubin estimator.

LDA is included as an **exploratory supervised projection**. It is not a convergence metric. Here it is used to replace PCA-style projection plots by asking whether prediction draws or trace features are separable by `swap_sigma` or by method.

## 1. Setup

The notebook assumes it is run from the repository root, e.g.

```bash
cd /root/bart-playground
jupyter notebook
```

Default store path:

```text
diagnosis/fixed100_testpoints_v2/store
```

If your store folder is elsewhere, edit `STORE_DIR` below.

In [ ]:
from __future__ import annotations

import ast
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error

try:
    import arviz as az
    HAS_ARVIZ = True
except Exception:
    HAS_ARVIZ = False
    warnings.warn("ArviZ not available. Falling back to classical R-hat and simple ESS.")

STORE_DIR = Path("diagnosis/fixed100_testpoints_v2/store")

DATASETS = ["Abalone", "Concrete", "Friedman"]
SWAP_LABELS = {
    "true": "swap_sigma=True",
    "false": "swap_sigma=False",
}
METHODS = ["default", "default_pt", "mtmh", "mtmh_pt"]

print("STORE_DIR =", STORE_DIR.resolve())
print("ArviZ available:", HAS_ARVIZ)

## 2. Robust CSV loaders

The experiment scripts save arrays using `np.savetxt` with a header containing the original shape:

```text
# original_shape=(...)
```

The helper below reconstructs the original array shape. This matters because `preds` should usually load as:

\[
(\text{n_chains}, \text{n_test_points}, \text{n_draws})
\]

and `trace_features` should usually load as:

\[
(\text{n_chains}, \text{n_draws}, \text{n_features})
\]

In [ ]:
def parse_shape_from_header(csv_path: Path):
    with open(csv_path, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    first_line = first_line.lstrip("#").strip()
    if not first_line.startswith("original_shape="):
        raise ValueError(f"Missing original_shape header in {csv_path}")
    return ast.literal_eval(first_line.split("=", 1)[1].strip())


def load_saved_numeric_csv(csv_path: Path) -> np.ndarray:
    csv_path = Path(csv_path)
    shape = parse_shape_from_header(csv_path)
    arr2d = np.loadtxt(csv_path, delimiter=",", comments="#")
    if arr2d.size == 0:
        return np.array([]).reshape(shape)
    if arr2d.ndim == 0:
        arr2d = arr2d.reshape(1, 1)
    elif arr2d.ndim == 1:
        # np.loadtxt returns 1D for one-row or one-column files.
        # Since _save_numeric_csv wrote a 2D array, this reshape is safe before restoring original shape.
        arr2d = arr2d.reshape(1, -1)
    return arr2d.reshape(shape)


def dataset_tag(dataset: str, swap_key: str) -> str:
    return f"fixed100_{dataset}__swap_sigma_{swap_key}"


def result_path(dataset: str, swap_key: str, subdir: str, run_id: int, method: str, metric: str) -> Path:
    tag = dataset_tag(dataset, swap_key)
    return STORE_DIR / tag / subdir / f"{tag}__run{run_id:03d}__{method}__{metric}.csv"


def discover_runs(dataset: str, swap_key: str) -> list[int]:
    tag = dataset_tag(dataset, swap_key)
    rmse_dir = STORE_DIR / tag / "subsample_rmse"
    if not rmse_dir.exists():
        return []
    pattern = re.compile(rf"^{re.escape(tag)}__run(?P<run>\d{{3}})__default__subsample_rmse\.csv$")
    runs = []
    for p in rmse_dir.glob(f"{tag}__run*__default__subsample_rmse.csv"):
        m = pattern.match(p.name)
        if m:
            runs.append(int(m.group("run")))
    return sorted(runs)


def check_completeness():
    rows = []
    for dataset in DATASETS:
        for swap_key in SWAP_LABELS:
            runs = discover_runs(dataset, swap_key)
            for method in METHODS:
                exists_all = True
                for run_id in runs:
                    p = result_path(dataset, swap_key, "subsample_rmse", run_id, method, "subsample_rmse")
                    exists_all = exists_all and p.exists()
                rows.append({
                    "dataset": dataset,
                    "swap_key": swap_key,
                    "swap_sigma": SWAP_LABELS[swap_key],
                    "runs_detected": runs,
                    "n_runs": len(runs),
                    "method": method,
                    "rmse_files_present": exists_all if runs else False,
                })
    return pd.DataFrame(rows)

completeness = check_completeness()
display(completeness)

## 3. Performance summary: RMSE, CRPS, coverage

For each dataset, method, and swap setting, we average over runs and chains.

- `subsample_rmse`: chain-level RMSE of the posterior mean prediction.
- `subsample_crps`: chain-level CRPS. In the patched script this is based on posterior predictive samples, so it matches the sampling-based CRPS definition:

\[
\mathrm{CRPS}=\mathbb{E}|X-y|-\frac{1}{2}\mathbb{E}|X-X'|
\]

- `coverage`: empirical 95% prediction interval coverage over the fixed 100 test points.

In [ ]:
def load_scalar_metric_values(dataset: str, swap_key: str, method: str, metric: str) -> np.ndarray:
    values = []
    for run_id in discover_runs(dataset, swap_key):
        p = result_path(dataset, swap_key, metric, run_id, method, metric)
        if not p.exists():
            continue
        arr = load_saved_numeric_csv(p)
        values.extend(np.asarray(arr).reshape(-1).tolist())
    return np.asarray(values, dtype=float)


def load_coverage_mean_values(dataset: str, swap_key: str, method: str) -> np.ndarray:
    values = []
    for run_id in discover_runs(dataset, swap_key):
        p = result_path(dataset, swap_key, "coverage", run_id, method, "coverage")
        if not p.exists():
            continue
        arr = load_saved_numeric_csv(p)
        # arr is usually [n_chains, n_test_points].
        if arr.ndim == 1:
            values.append(float(np.mean(arr)))
        else:
            values.extend(np.mean(arr, axis=tuple(range(1, arr.ndim))).reshape(-1).tolist())
    return np.asarray(values, dtype=float)


summary_rows = []
for dataset in DATASETS:
    for swap_key in SWAP_LABELS:
        for method in METHODS:
            rmse_vals = load_scalar_metric_values(dataset, swap_key, method, "subsample_rmse")
            crps_vals = load_scalar_metric_values(dataset, swap_key, method, "subsample_crps")
            cov_vals = load_coverage_mean_values(dataset, swap_key, method)
            summary_rows.append({
                "dataset": dataset,
                "swap_sigma": SWAP_LABELS[swap_key],
                "method": method,
                "n_rmse_values": rmse_vals.size,
                "rmse_mean": np.nanmean(rmse_vals) if rmse_vals.size else np.nan,
                "rmse_sd": np.nanstd(rmse_vals, ddof=1) if rmse_vals.size > 1 else np.nan,
                "crps_mean": np.nanmean(crps_vals) if crps_vals.size else np.nan,
                "crps_sd": np.nanstd(crps_vals, ddof=1) if crps_vals.size > 1 else np.nan,
                "coverage_mean": np.nanmean(cov_vals) if cov_vals.size else np.nan,
                "coverage_sd": np.nanstd(cov_vals, ddof=1) if cov_vals.size > 1 else np.nan,
            })

perf_summary = pd.DataFrame(summary_rows)
display(perf_summary.sort_values(["dataset", "method", "swap_sigma"]))

### Performance comparison table

This pivot is useful for reporting. Lower RMSE / CRPS is better. Coverage is not a loss: it should be interpreted relative to the nominal level, here approximately 0.95 for a calibrated 95% predictive interval.

In [ ]:
for metric in ["rmse_mean", "crps_mean", "coverage_mean"]:
    print("\n===", metric, "===")
    pivot = perf_summary.pivot_table(
        index=["dataset", "method"],
        columns="swap_sigma",
        values=metric,
        aggfunc="mean",
    )
    display(pivot)

## 4. Trace loading

For diagnostics, we need per-draw traces.

The most important trace for this task is:

\[
\sigma_t^2
\]

because the code change directly concerns whether sigma is swapped during PT.

The patched script also saves `trace_features`, including `eps_sigma2`, RMSE, tree depth/leaves, and split counts if available.

In [ ]:
def load_trace_features(dataset: str, swap_key: str, run_id: int, method: str):
    p = result_path(dataset, swap_key, "trace_features", run_id, method, "trace_features")
    if not p.exists():
        return None, None
    arr = load_saved_numeric_csv(p)
    col_path = result_path(dataset, swap_key, "trace_feature_columns", run_id, method, "trace_feature_columns")
    columns = None
    if col_path.exists():
        # column file is saved as object csv with JSON strings in second column
        df = pd.read_csv(col_path)
        if "value_json" in df.columns:
            columns = [json_val.strip('"') if isinstance(json_val, str) else str(json_val)
                       for json_val in df["value_json"].tolist()]
            # If the JSON strings have escaped content, parse them properly.
            import json
            columns = [json.loads(v) if isinstance(v, str) else str(v) for v in df["value_json"].tolist()]
    if columns is None:
        columns = [f"feature_{i}" for i in range(arr.shape[-1])]
    return arr, columns


def load_sigmas(dataset: str, swap_key: str, run_id: int, method: str) -> np.ndarray | None:
    # Prefer trace_features if available and contains eps_sigma2.
    arr, cols = load_trace_features(dataset, swap_key, run_id, method)
    if arr is not None and "eps_sigma2" in cols:
        idx = cols.index("eps_sigma2")
        return np.asarray(arr)[..., idx]

    # Fallback to saved sigmas.
    p = result_path(dataset, swap_key, "sigmas", run_id, method, "sigmas")
    if not p.exists():
        return None
    sig = load_saved_numeric_csv(p)
    return np.asarray(sig).squeeze()


# Quick shape check.
shape_rows = []
for dataset in DATASETS:
    for swap_key in SWAP_LABELS:
        for method in METHODS:
            for run_id in discover_runs(dataset, swap_key):
                sig = load_sigmas(dataset, swap_key, run_id, method)
                arr, cols = load_trace_features(dataset, swap_key, run_id, method)
                shape_rows.append({
                    "dataset": dataset,
                    "swap_sigma": SWAP_LABELS[swap_key],
                    "method": method,
                    "run": run_id,
                    "sigma_shape": None if sig is None else tuple(np.asarray(sig).shape),
                    "trace_features_shape": None if arr is None else tuple(np.asarray(arr).shape),
                    "n_trace_feature_columns": None if cols is None else len(cols),
                })
shape_df = pd.DataFrame(shape_rows)
display(shape_df)

## 5. Autocorrelation and ESS definitions

For a scalar chain \(X_t\), lag-\(k\) autocorrelation is:

\[
\rho_k=\frac{\mathrm{Cov}(X_t, X_{t+k})}{\mathrm{Var}(X_t)}.
\]

For a finite chain, we estimate this by centering the sequence and computing normalized lagged inner products.

ESS is estimated as:

\[
\mathrm{ESS}=\frac{N}{1+2\sum_{k=1}^{K}\rho_k},
\]

where \(K\) is truncated when the autocorrelation sequence becomes non-positive. If ArviZ is installed, we use ArviZ ESS for reporting because it is more robust. The manual ESS is kept as a transparent fallback.

In [ ]:
def autocorr_1d(x: np.ndarray, max_lag: int = 200) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float).reshape(-1)
    x = x[np.isfinite(x)]
    n = x.size
    if n < 2:
        return np.array([1.0]), np.array([0])
    x = x - np.mean(x)
    denom = float(np.dot(x, x))
    if denom <= 0:
        lags = np.arange(min(max_lag, n - 1) + 1)
        return np.ones_like(lags, dtype=float), lags
    max_lag = min(int(max_lag), n - 1)
    ac = np.empty(max_lag + 1, dtype=float)
    for lag in range(max_lag + 1):
        ac[lag] = np.dot(x[: n - lag], x[lag:]) / denom
    return ac, np.arange(max_lag + 1)


def ess_from_acf(ac: np.ndarray, n: int) -> float:
    # Positive-sequence truncation: stop at first non-positive autocorrelation after lag 0.
    ac = np.asarray(ac, dtype=float)
    if ac.size <= 1:
        return float(n)
    positive = []
    for rho in ac[1:]:
        if not np.isfinite(rho) or rho <= 0:
            break
        positive.append(float(rho))
    tau = 1.0 + 2.0 * np.sum(positive)
    if tau <= 0:
        return float(n)
    return float(n / tau)


def classical_rhat(chains: np.ndarray) -> float:
    # chains shape: [m, n]
    chains = np.asarray(chains, dtype=float)
    if chains.ndim != 2:
        raise ValueError("classical_rhat expects shape [n_chains, n_draws].")
    m, n = chains.shape
    if m < 2 or n < 2:
        return np.nan
    chain_means = np.mean(chains, axis=1)
    chain_vars = np.var(chains, axis=1, ddof=1)
    W = np.mean(chain_vars)
    B = n * np.var(chain_means, ddof=1)
    if W <= 0:
        return np.nan
    var_hat = ((n - 1) / n) * W + (1 / n) * B
    return float(np.sqrt(var_hat / W))


def rhat_for_chains(chains: np.ndarray) -> float:
    # chains shape: [n_chains, n_draws]
    chains = np.asarray(chains, dtype=float)
    if HAS_ARVIZ:
        try:
            return float(az.rhat(chains))
        except Exception:
            pass
    return classical_rhat(chains)


def ess_for_chains(chains: np.ndarray) -> float:
    # chains shape: [n_chains, n_draws]
    chains = np.asarray(chains, dtype=float)
    if HAS_ARVIZ:
        try:
            return float(az.ess(chains))
        except Exception:
            pass
    # fallback: compute per-chain ESS and sum them
    total = 0.0
    for c in range(chains.shape[0]):
        ac, _ = autocorr_1d(chains[c], max_lag=min(500, chains.shape[1] - 1))
        total += ess_from_acf(ac, chains.shape[1])
    return float(total)

## 6. Sigma peak diagnostics

The core question from the code change is whether `swap_sigma=True` creates abnormal peaks in the \(\sigma^2\) trace.

We summarize each chain using:

- `sigma_mean`
- `sigma_median`
- `sigma_max`
- `peak_ratio = max(sigma2) / median(sigma2)`

A large peak ratio means the chain contains extreme sigma excursions. This directly tests the phenomenon observed in the preliminary Concrete run.

In [ ]:
peak_rows = []

for dataset in DATASETS:
    for swap_key in SWAP_LABELS:
        for method in METHODS:
            for run_id in discover_runs(dataset, swap_key):
                sig = load_sigmas(dataset, swap_key, run_id, method)
                if sig is None:
                    continue
                sig = np.asarray(sig, dtype=float)
                if sig.ndim == 1:
                    sig = sig.reshape(1, -1)
                elif sig.ndim > 2:
                    sig = sig.reshape(sig.shape[0], -1)
                for chain_id in range(sig.shape[0]):
                    x = sig[chain_id]
                    med = np.nanmedian(x)
                    peak_rows.append({
                        "dataset": dataset,
                        "swap_sigma": SWAP_LABELS[swap_key],
                        "method": method,
                        "run": run_id,
                        "chain": chain_id,
                        "sigma_mean": np.nanmean(x),
                        "sigma_median": med,
                        "sigma_max": np.nanmax(x),
                        "peak_ratio_max_over_median": np.nanmax(x) / med if med > 0 else np.nan,
                    })

peak_df = pd.DataFrame(peak_rows)
display(peak_df.sort_values(["dataset", "method", "swap_sigma", "run", "chain"]))

print("Mean peak ratio by dataset/method/swap:")
display(
    peak_df.groupby(["dataset", "method", "swap_sigma"], as_index=False)
    ["peak_ratio_max_over_median"].mean()
    .sort_values(["dataset", "method", "swap_sigma"])
)

### Plot sigma traces

Each plot overlays chains for `swap_sigma=True` and `swap_sigma=False`.

The cleanest comparison is usually for the PT methods:

- `default_pt`
- `mtmh_pt`

because `swap_sigma` only affects PT swap behavior.

In [ ]:
def plot_sigma_traces(dataset: str, method: str, run_id: int = 0, max_draws: int | None = None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
    for ax, swap_key in zip(axes, ["true", "false"]):
        sig = load_sigmas(dataset, swap_key, run_id, method)
        if sig is None:
            ax.set_title(f"{dataset} | {method} | {SWAP_LABELS[swap_key]} | missing")
            continue
        sig = np.asarray(sig, dtype=float)
        if sig.ndim == 1:
            sig = sig.reshape(1, -1)
        elif sig.ndim > 2:
            sig = sig.reshape(sig.shape[0], -1)
        if max_draws is not None:
            sig = sig[:, :max_draws]
        for chain_id in range(sig.shape[0]):
            ax.plot(sig[chain_id], linewidth=0.8, alpha=0.8, label=f"chain {chain_id}")
        ax.set_title(f"{dataset} | {method} | {SWAP_LABELS[swap_key]} | run {run_id:03d}")
        ax.set_xlabel("iteration")
        ax.set_ylabel(r"$\sigma^2$")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


for dataset in DATASETS:
    for method in ["default_pt", "mtmh_pt"]:
        runs = discover_runs(dataset, "true")
        if runs:
            plot_sigma_traces(dataset, method, run_id=runs[0])

## 7. Sigma autocorrelation, ESS, and R-hat

For each dataset/method/swap setting, we compute:

- ACF for each chain's \(\sigma^2\)
- ESS using ArviZ when available
- R-hat using ArviZ rank-normalized R-hat when available

Interpretation:

- Faster ACF decay is better.
- Higher ESS is better.
- \(\hat R \approx 1\) is better.

In [ ]:
diag_rows = []

for dataset in DATASETS:
    for swap_key in SWAP_LABELS:
        for method in METHODS:
            # Pool runs separately in reporting. Do not concatenate runs into one chain for R-hat.
            for run_id in discover_runs(dataset, swap_key):
                sig = load_sigmas(dataset, swap_key, run_id, method)
                if sig is None:
                    continue
                sig = np.asarray(sig, dtype=float)
                if sig.ndim == 1:
                    sig = sig.reshape(1, -1)
                elif sig.ndim > 2:
                    sig = sig.reshape(sig.shape[0], -1)
                diag_rows.append({
                    "dataset": dataset,
                    "swap_sigma": SWAP_LABELS[swap_key],
                    "method": method,
                    "run": run_id,
                    "n_chains": sig.shape[0],
                    "n_draws": sig.shape[1],
                    "sigma_ess": ess_for_chains(sig),
                    "sigma_rhat": rhat_for_chains(sig),
                })

sigma_diag_df = pd.DataFrame(diag_rows)
display(sigma_diag_df.sort_values(["dataset", "method", "swap_sigma", "run"]))

print("Average sigma ESS/R-hat by dataset/method/swap:")
display(
    sigma_diag_df.groupby(["dataset", "method", "swap_sigma"], as_index=False)
    .agg(
        sigma_ess_mean=("sigma_ess", "mean"),
        sigma_ess_sd=("sigma_ess", "std"),
        sigma_rhat_mean=("sigma_rhat", "mean"),
        sigma_rhat_max=("sigma_rhat", "max"),
    )
    .sort_values(["dataset", "method", "swap_sigma"])
)

In [ ]:
def plot_sigma_acf(dataset: str, method: str, run_id: int = 0, max_lag: int = 200):
    fig, ax = plt.subplots(figsize=(9, 4))
    for swap_key in ["true", "false"]:
        sig = load_sigmas(dataset, swap_key, run_id, method)
        if sig is None:
            continue
        sig = np.asarray(sig, dtype=float)
        if sig.ndim == 1:
            sig = sig.reshape(1, -1)
        elif sig.ndim > 2:
            sig = sig.reshape(sig.shape[0], -1)
        acfs = []
        for chain_id in range(sig.shape[0]):
            ac, lags = autocorr_1d(sig[chain_id], max_lag=max_lag)
            acfs.append(ac)
        min_len = min(len(a) for a in acfs)
        acf_mean = np.mean([a[:min_len] for a in acfs], axis=0)
        ax.plot(lags[:min_len], acf_mean, linewidth=1.5, label=SWAP_LABELS[swap_key])
    ax.axhline(0.0, linewidth=0.8)
    ax.set_title(f"{dataset} | {method} | sigma autocorrelation | run {run_id:03d}")
    ax.set_xlabel("lag")
    ax.set_ylabel("autocorrelation")
    ax.legend()
    plt.tight_layout()
    plt.show()


for dataset in DATASETS:
    for method in ["default_pt", "mtmh_pt"]:
        runs = discover_runs(dataset, "true")
        if runs:
            plot_sigma_acf(dataset, method, run_id=runs[0], max_lag=200)

## 8. Trace-feature diagnostics beyond sigma

If `trace_features` are available, we can also inspect tree-structure traces:

- average leaves
- average depth
- total split count

This helps distinguish whether a sigma peak is associated with abnormal tree structures or only with \(\sigma^2\).

In [ ]:
def get_trace_feature_series(dataset: str, swap_key: str, run_id: int, method: str, feature_name: str):
    arr, cols = load_trace_features(dataset, swap_key, run_id, method)
    if arr is None or feature_name not in cols:
        return None
    idx = cols.index(feature_name)
    x = np.asarray(arr)[..., idx]
    if x.ndim == 1:
        x = x.reshape(1, -1)
    elif x.ndim > 2:
        x = x.reshape(x.shape[0], -1)
    return x


candidate_features = ["rmse", "avg_leaves", "avg_depth", "total_splits"]
available_feature_rows = []

for dataset in DATASETS:
    for swap_key in SWAP_LABELS:
        for method in METHODS:
            runs = discover_runs(dataset, swap_key)
            if not runs:
                continue
            arr, cols = load_trace_features(dataset, swap_key, runs[0], method)
            available_feature_rows.append({
                "dataset": dataset,
                "swap_sigma": SWAP_LABELS[swap_key],
                "method": method,
                "available_columns": cols,
            })

available_features_df = pd.DataFrame(available_feature_rows)
display(available_features_df)

In [ ]:
feature_diag_rows = []

for dataset in DATASETS:
    for swap_key in SWAP_LABELS:
        for method in METHODS:
            for run_id in discover_runs(dataset, swap_key):
                for feature_name in candidate_features:
                    x = get_trace_feature_series(dataset, swap_key, run_id, method, feature_name)
                    if x is None:
                        continue
                    feature_diag_rows.append({
                        "dataset": dataset,
                        "swap_sigma": SWAP_LABELS[swap_key],
                        "method": method,
                        "run": run_id,
                        "feature": feature_name,
                        "ess": ess_for_chains(x),
                        "rhat": rhat_for_chains(x),
                        "mean": np.nanmean(x),
                    })

feature_diag_df = pd.DataFrame(feature_diag_rows)
display(feature_diag_df.sort_values(["dataset", "feature", "method", "swap_sigma", "run"]))

print("Average feature diagnostics:")
display(
    feature_diag_df.groupby(["dataset", "feature", "method", "swap_sigma"], as_index=False)
    .agg(ess_mean=("ess", "mean"), rhat_mean=("rhat", "mean"), rhat_max=("rhat", "max"), mean_value=("mean", "mean"))
    .sort_values(["dataset", "feature", "method", "swap_sigma"])
)

## 9. LDA as a supervised projection

LDA is not a convergence metric. It is a supervised dimension-reduction method.

For two groups, LDA finds a direction

\[
w \propto \Sigma^{-1}(\mu_1-\mu_0)
\]

that separates the two groups as much as possible relative to within-group variance.

Here we use LDA to replace PCA-style exploratory plots.

Two useful LDA questions:

1. **Swap separation:** for a fixed dataset and method, can LDA separate draws from `swap_sigma=True` and `swap_sigma=False`?
2. **Method separation:** for a fixed dataset and swap setting, can LDA separate the four methods?

If the LDA projection shows strong separation, it means the posterior draws / trace features differ systematically across groups. It does **not** alone prove better mixing.

### 9.1 Prediction-vector LDA

Each posterior draw is represented by its prediction vector on the fixed 100 test points:

\[
x_t = (f_t(x_1), f_t(x_2), \ldots, f_t(x_{100}))
\]

So each MCMC draw becomes one row in a matrix.

This is similar in spirit to PCA-on-predictions, but LDA uses labels such as `swap_sigma=True/False` or method labels.

In [ ]:
def load_preds(dataset: str, swap_key: str, run_id: int, method: str) -> np.ndarray | None:
    p = result_path(dataset, swap_key, "preds", run_id, method, "preds")
    if not p.exists():
        return None
    arr = load_saved_numeric_csv(p)
    # Expected shape [n_chains, n_test, n_draws].
    if arr.ndim != 3:
        raise ValueError(f"Unexpected preds shape {arr.shape} for {p}")
    return arr


def preds_to_draw_matrix(preds: np.ndarray, max_draws_per_chain: int | None = None):
    # [n_chains, n_test, n_draws] -> [n_chains*n_draws, n_test]
    preds = np.asarray(preds, dtype=float)
    if max_draws_per_chain is not None:
        preds = preds[:, :, :max_draws_per_chain]
    n_chains, n_test, n_draws = preds.shape
    X = preds.transpose(0, 2, 1).reshape(n_chains * n_draws, n_test)
    chain_ids = np.repeat(np.arange(n_chains), n_draws)
    draw_ids = np.tile(np.arange(n_draws), n_chains)
    return X, chain_ids, draw_ids


def lda_projection(X: np.ndarray, y: np.ndarray, n_components: int | None = None):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    n_classes = len(np.unique(y))
    max_components = min(n_classes - 1, X.shape[1])
    if n_components is None:
        n_components = max_components
    else:
        n_components = min(n_components, max_components)
    if n_components < 1:
        raise ValueError("LDA needs at least two classes.")
    # StandardScaler avoids domination by high-scale dimensions.
    model = make_pipeline(
        StandardScaler(),
        LinearDiscriminantAnalysis(n_components=n_components, solver="svd"),
    )
    Z = model.fit_transform(X, y)
    return Z, model


def build_prediction_lda_swap_data(dataset: str, method: str, run_id: int, max_draws_per_chain: int | None = None):
    X_parts, y_parts, meta_parts = [], [], []
    for swap_key in ["true", "false"]:
        preds = load_preds(dataset, swap_key, run_id, method)
        if preds is None:
            continue
        X, chain_ids, draw_ids = preds_to_draw_matrix(preds, max_draws_per_chain=max_draws_per_chain)
        X_parts.append(X)
        y_parts.append(np.array([SWAP_LABELS[swap_key]] * X.shape[0]))
        meta_parts.append(pd.DataFrame({
            "swap_sigma": SWAP_LABELS[swap_key],
            "chain": chain_ids,
            "draw": draw_ids,
        }))
    if not X_parts:
        return None, None, None
    return np.vstack(X_parts), np.concatenate(y_parts), pd.concat(meta_parts, ignore_index=True)


def plot_lda_1d_by_group(Z, y, title: str):
    z = np.asarray(Z)[:, 0]
    df = pd.DataFrame({"LD1": z, "group": y})
    fig, ax = plt.subplots(figsize=(9, 4))
    groups = list(pd.unique(df["group"]))
    for group in groups:
        vals = df.loc[df["group"] == group, "LD1"].values
        ax.hist(vals, bins=40, alpha=0.45, density=True, label=str(group))
    ax.set_title(title)
    ax.set_xlabel("LD1")
    ax.set_ylabel("density")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_lda_trace_by_group(Z, y, meta, title: str):
    z = np.asarray(Z)[:, 0]
    df = meta.copy()
    df["LD1"] = z
    df["group"] = y
    fig, ax = plt.subplots(figsize=(10, 4))
    for group in pd.unique(df["group"]):
        # Plot mean LD1 over chains at each draw index to avoid overcrowding.
        sub = df[df["group"] == group]
        mean_trace = sub.groupby("draw")["LD1"].mean()
        ax.plot(mean_trace.index, mean_trace.values, linewidth=1.2, label=str(group))
    ax.set_title(title)
    ax.set_xlabel("draw")
    ax.set_ylabel("mean LD1 over chains")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Swap LDA on prediction vectors: focus on PT methods because swap_sigma only affects PT.
for dataset in DATASETS:
    for method in ["default_pt", "mtmh_pt"]:
        runs = discover_runs(dataset, "true")
        if not runs:
            continue
        run_id = runs[0]
        out = build_prediction_lda_swap_data(dataset, method, run_id, max_draws_per_chain=None)
        if out[0] is None:
            continue
        X, y, meta = out
        Z, lda_model = lda_projection(X, y, n_components=1)
        plot_lda_1d_by_group(Z, y, f"{dataset} | {method} | Prediction-vector LDA: swap_sigma True vs False | run {run_id:03d}")
        plot_lda_trace_by_group(Z, y, meta, f"{dataset} | {method} | LD1 trace: swap_sigma True vs False | run {run_id:03d}")

### 9.2 Method-separation LDA

Here the label is the method:

\[
\text{method} \in \{\text{default}, \text{default\_pt}, \text{mtmh}, \text{mtmh\_pt}\}
\]

For four classes, LDA can have up to three discriminant axes. We plot LD1 vs LD2 when available.

In [ ]:
def build_prediction_lda_method_data(dataset: str, swap_key: str, run_id: int, max_draws_per_chain: int | None = None):
    X_parts, y_parts, meta_parts = [], [], []
    for method in METHODS:
        preds = load_preds(dataset, swap_key, run_id, method)
        if preds is None:
            continue
        X, chain_ids, draw_ids = preds_to_draw_matrix(preds, max_draws_per_chain=max_draws_per_chain)
        X_parts.append(X)
        y_parts.append(np.array([method] * X.shape[0]))
        meta_parts.append(pd.DataFrame({
            "method": method,
            "chain": chain_ids,
            "draw": draw_ids,
        }))
    if not X_parts:
        return None, None, None
    return np.vstack(X_parts), np.concatenate(y_parts), pd.concat(meta_parts, ignore_index=True)


def plot_lda_2d_by_group(Z, y, title: str):
    Z = np.asarray(Z)
    fig, ax = plt.subplots(figsize=(8, 6))
    groups = list(pd.unique(pd.Series(y)))
    if Z.shape[1] == 1:
        for group in groups:
            vals = Z[np.asarray(y) == group, 0]
            ax.hist(vals, bins=40, alpha=0.45, density=True, label=str(group))
        ax.set_xlabel("LD1")
        ax.set_ylabel("density")
    else:
        for group in groups:
            pts = Z[np.asarray(y) == group]
            ax.scatter(pts[:, 0], pts[:, 1], s=8, alpha=0.35, label=str(group))
        ax.set_xlabel("LD1")
        ax.set_ylabel("LD2")
    ax.set_title(title)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


for dataset in DATASETS:
    for swap_key in ["true", "false"]:
        runs = discover_runs(dataset, swap_key)
        if not runs:
            continue
        run_id = runs[0]
        out = build_prediction_lda_method_data(dataset, swap_key, run_id, max_draws_per_chain=None)
        if out[0] is None:
            continue
        X, y, meta = out
        Z, lda_model = lda_projection(X, y, n_components=2)
        plot_lda_2d_by_group(Z, y, f"{dataset} | {SWAP_LABELS[swap_key]} | Prediction-vector LDA by method | run {run_id:03d}")

## 10. Trace-feature LDA

Prediction-vector LDA asks whether the fitted function differs between groups.

Trace-feature LDA asks whether internal sampler state summaries differ between groups, using features like:

\[
(\sigma^2, \mathrm{RMSE}, \text{avg leaves}, \text{avg depth}, \text{split counts})
\]

This is closer to diagnosing whether `swap_sigma=True` changes sampler state behavior.

In [ ]:
def trace_features_to_matrix(arr: np.ndarray, max_draws_per_chain: int | None = None):
    # Expected arr [n_chains, n_draws, n_features].
    arr = np.asarray(arr, dtype=float)
    if arr.ndim != 3:
        raise ValueError(f"Expected trace_features shape [chains, draws, features], got {arr.shape}")
    if max_draws_per_chain is not None:
        arr = arr[:, :max_draws_per_chain, :]
    n_chains, n_draws, n_features = arr.shape
    X = arr.reshape(n_chains * n_draws, n_features)
    chain_ids = np.repeat(np.arange(n_chains), n_draws)
    draw_ids = np.tile(np.arange(n_draws), n_chains)
    return X, chain_ids, draw_ids


def build_trace_lda_swap_data(dataset: str, method: str, run_id: int, max_draws_per_chain: int | None = None):
    X_parts, y_parts, meta_parts = [], [], []
    columns_ref = None
    for swap_key in ["true", "false"]:
        arr, cols = load_trace_features(dataset, swap_key, run_id, method)
        if arr is None:
            continue
        if columns_ref is None:
            columns_ref = cols
        X, chain_ids, draw_ids = trace_features_to_matrix(arr, max_draws_per_chain=max_draws_per_chain)
        X_parts.append(X)
        y_parts.append(np.array([SWAP_LABELS[swap_key]] * X.shape[0]))
        meta_parts.append(pd.DataFrame({
            "swap_sigma": SWAP_LABELS[swap_key],
            "chain": chain_ids,
            "draw": draw_ids,
        }))
    if not X_parts:
        return None, None, None, None
    X_all = np.vstack(X_parts)
    # Remove columns with zero variance or non-finite values.
    finite_cols = np.all(np.isfinite(X_all), axis=0)
    var_cols = np.nanstd(X_all, axis=0) > 0
    keep = finite_cols & var_cols
    return X_all[:, keep], np.concatenate(y_parts), pd.concat(meta_parts, ignore_index=True), [c for c, k in zip(columns_ref, keep) if k]


for dataset in DATASETS:
    for method in ["default_pt", "mtmh_pt"]:
        runs = discover_runs(dataset, "true")
        if not runs:
            continue
        run_id = runs[0]
        out = build_trace_lda_swap_data(dataset, method, run_id, max_draws_per_chain=None)
        if out[0] is None:
            continue
        X, y, meta, kept_cols = out
        print(f"{dataset} | {method} | kept trace feature columns:", kept_cols)
        Z, lda_model = lda_projection(X, y, n_components=1)
        plot_lda_1d_by_group(Z, y, f"{dataset} | {method} | Trace-feature LDA: swap_sigma True vs False | run {run_id:03d}")
        plot_lda_trace_by_group(Z, y, meta, f"{dataset} | {method} | Trace-feature LD1 trace | run {run_id:03d}")

## 11. Suggested reporting logic

Use the notebook results in this order:

1. **Performance table**: RMSE / CRPS / coverage for all datasets, methods, and swap settings.
2. **Sigma peak table**: especially `peak_ratio = max(sigma2) / median(sigma2)`.
3. **Sigma trace plots**: visually show whether `swap_sigma=True` has large peaks.
4. **ACF / ESS / R-hat**: formally assess mixing of \(\sigma^2\) and state features.
5. **LDA plots**: exploratory evidence that `swap_sigma=True` and `False` produce different draw distributions.

Important wording:

- ACF, ESS, and R-hat are MCMC diagnostics.
- LDA is an exploratory supervised projection, not a convergence metric.
- If `swap_sigma=True` has large sigma peak ratios and worse ESS/R-hat, that supports the observation that swapping sigma produces problematic behavior.